# House Price Prediction — Model Training & Comparison

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

RANDOM_STATE = 42
TEST_SIZE = 0.2
DATA_PATH = "../data/USA_Housing.csv"
MODEL_OUTPUT_PATH = "../models/best_model.pkl"

FEATURES = [
    "Avg. Area Income",
    "Avg. Area House Age",
    "Avg. Area Number of Rooms",
    "Avg. Area Number of Bedrooms",
    "Area Population",
]
TARGET = "Price"

## 1. Load Data and Split (80/20)

In [2]:
df = pd.read_csv(DATA_PATH)

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4000, 5)
Test shape: (1000, 5)


## 2. Train Multiple Models (with Pipelines)

In [3]:
def build_pipeline(model, degree=None):
    """Builds a scikit-learn Pipeline with optional polynomial expansion,
    scaling, and a given estimator."""
    steps = []
    if degree is not None:
        steps.append(("poly", PolynomialFeatures(degree=degree, include_bias=False)))
    steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)


candidate_pipelines = {
    "Linear Regression": build_pipeline(LinearRegression()),
    "Ridge (alpha=1.0)": build_pipeline(Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    "Lasso (alpha=0.1)": build_pipeline(Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=10000)),
    "Polynomial(deg=2) + Ridge": build_pipeline(Ridge(alpha=1.0, random_state=RANDOM_STATE), degree=2),
    "KNN (k=5)": build_pipeline(KNeighborsRegressor(n_neighbors=5)),
    "KNN (k=9)": build_pipeline(KNeighborsRegressor(n_neighbors=9)),
}

results = []
fitted_pipelines = {}

for name, pipeline in candidate_pipelines.items():
    pipeline.fit(X_train, y_train)

    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    test_mse = mean_squared_error(y_test, test_pred)

    results.append({
        "Model": name,
        "Train R2": round(train_r2, 4),
        "Test R2": round(test_r2, 4),
        "Test MSE": round(test_mse, 2),
    })
    fitted_pipelines[name] = pipeline

results_df = pd.DataFrame(results)
results_df

,Model,Train R2,Test R2,Test MSE
0,Linear Regression,0.9180,0.9180,1.008901e+10
1,Ridge (alpha=1.0),0.9180,0.9180,1.008900e+10
2,Lasso (alpha=0.1),0.9180,0.9180,1.008901e+10
3,Polynomial(deg=2) + Ridge,0.9181,0.9179,1.010548e+10
4,KNN (k=5),0.9109,0.8693,1.607824e+10
5,KNN (k=9),0.9011,0.8756,1.531123e+10


## 3. Model Comparison Table (sorted by Test R²)

In [4]:
comparison_table = results_df.sort_values(by="Test R2", ascending=False).reset_index(drop=True)
comparison_table

,Model,Train R2,Test R2,Test MSE
0,Linear Regression,0.9180,0.9180,1.008901e+10
1,Ridge (alpha=1.0),0.9180,0.9180,1.008900e+10
2,Lasso (alpha=0.1),0.9180,0.9180,1.008901e+10
3,Polynomial(deg=2) + Ridge,0.9181,0.9179,1.010548e+10
4,KNN (k=9),0.9011,0.8756,1.531123e+10
5,KNN (k=5),0.9109,0.8693,1.607824e+10


## 4. Select Best Model

In [5]:
best_model_name = comparison_table.iloc[0]["Model"]
best_pipeline = fitted_pipelines[best_model_name]

overfit_gap = comparison_table.iloc[0]["Train R2"] - comparison_table.iloc[0]["Test R2"]

print(f"Best model: {best_model_name}")
print(comparison_table.iloc[0])
print(f"\nTrain-Test R2 gap (overfitting check): {overfit_gap:.4f}")
print("A small gap indicates the model generalizes well and is not overfitting.")

Best model: Linear Regression
Model        Linear Regression
Train R2                 0.918
Test R2                  0.918
Test MSE    10089009300.889999
Name: 0, dtype: object

Train-Test R2 gap (overfitting check): 0.0000
A small gap indicates the model generalizes well and is not overfitting.


## 5. Save the Best Pipeline

In [6]:
Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, MODEL_OUTPUT_PATH)
print(f"Saved best pipeline ('{best_model_name}') to {MODEL_OUTPUT_PATH}")

Saved best pipeline ('Linear Regression') to ../models/best_model.pkl
